# LedaFlow Simulations with PID Control

In [1]:
using CSV
using DataFrames
lf_case_id = "1706e3ca-b91b-4468-be4c-7d70e5b9bfbf" # Ledaflow Case ID  
pid_ctrl_file = "ctrl_scheme_C.jl"

include(pid_ctrl_file)
include("lf_softshell.jl")
include("run_ledaflow_sim_script.jl")

# RUN LEDAFOW TO STEADY-STATE 
include("run_ledaflow_ss.jl")

Process(`'/mnt/c/Program Files/Kongsberg/LedaFlow Engineering v2.11.271.018/softsh.exe' '/home/archanak/Kumaraswamy_2024_2027/Ongoing Work/2026_NPC_Workshop/ledaflow_ss.js'`, ProcessExited(0))

In [2]:
##############################
# INITIALIZATION 
##############################
# Initialization for outputs 
outputs = [1.0, 1.0, 1.0, 1.0, 3458.0]

# Initialization for measurements
#### List of measurements from LedaFlow (order MUST match run_ledaflow_sim)
#  1. Mainline Flow Rate (19500 m)     8. Well 2 BHP
#  2. Mainline Pressure (500 m)        9. Well 3 Flow Rate
#  3. Well 1 Flow Rate                10. Well 3 Pressure
#  4. Well 1 Pressure                 11. Well 3 BHP
#  5. Well 1 BHP                      12. Well 4 Flow Rate
#  6. Well 2 Flow Rate                13. Well 4 Pressure
#  7. Well 2 Pressure                 14. Well 4 BHP

ss_output_csv_path = joinpath(@__DIR__, "ss_lf_output.csv")

df = CSV.read(ss_output_csv_path, DataFrame;
        header = 10,                 # column names on row 10
        skipto = 12,                 # skip the units row (11)
        delim = ',',                 # comma-delimited export
        decimal = '.',               # period decimals (e.g. 6.0000000)
        missingstring = ["--", ""],  # LedaFlow missing marker + trailing empty col
        normalizenames = false,      # keep names like "Pressure@Line 1 P 500m"
        types = Float64,
        )

mainline_pressure = df[end, "Pressure@Mainline P 500m"]
mainline_flow     = df[end, "MFR - total@Mainline P 19500m"]*(-1)   # match sim (19500 m)

well1_flow = df[end, "MFR - total liquid@Wellbore1 P MFR"]*(-1)
well2_flow = df[end, "MFR - total liquid@Wellbore2 P MFR"]*(-1)
well3_flow = df[end, "MFR - total liquid@Wellbore3 P MFR"]*(-1)
well4_flow = df[end, "MFR - total liquid@Wellbore4 P MFR"]*(-1)

well1_press = df[end, "Pressure@Wellbore1 P MFR"]
well2_press = df[end, "Pressure@Wellbore2 P MFR"]
well3_press = df[end, "Pressure@Wellbore3 P MFR"]
well4_press = df[end, "Pressure@Wellbore4 P MFR"]

well1_bhp = df[end, "BHP - Zone 1@Well 1"]
well2_bhp = df[end, "BHP - Zone 1@Well 2"]
well3_bhp = df[end, "BHP - Zone 1@Well 3"]
well4_bhp = df[end, "BHP - Zone 1@Well 4"]

old_measurements = [mainline_flow, mainline_pressure,
    well1_flow, well1_press, well1_bhp,
    well2_flow, well2_press, well2_bhp,
    well3_flow, well3_press, well3_bhp,
    well4_flow, well4_press, well4_bhp]
old_outputs = outputs
old_clamped_outputs = outputs
clamped_outputs = outputs
println("Initial Measurements: $old_measurements")
println("Initial Outputs: $old_outputs")


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


Initial Measurements: [500.24592, 197.81419, 125.04508, 150.79354, 245.04503, 125.04508, 150.79354, 245.04503, 125.04508, 150.79354, 245.04503, 125.04508, 150.79354, 245.04503]
Initial Outputs: [1.0, 1.0, 1.0, 1.0, 3458.0]


In [3]:
##############################
# RUN PID CONTROLLER EVER TIME STEP 
##############################
# Time step in seconds 
dt = 30
nt = 7200

for i = 0:dt:nt
    new_measurements = run_ledaflow_sim(clamped_outputs, old_clamped_outputs, dt, i)    

    # Update old outputs and old clamped outputs for the next PID run 
    old_outputs = outputs
    old_clamped_outputs = clamped_outputs

    outputs, clamped_outputs = run_pid_controller(new_measurements, old_measurements, old_outputs, old_clamped_outputs, dt, i)

    # Update old measurements and outputs for next time step
    old_measurements = new_measurements
    println("Time step: $i")
end 



choke_vlv_op_1: 1.0, choke_vlv_op_2: 1.0, choke_vlv_op_3: 1.0, choke_vlv_op_4: 1.0, pump_speed: 3458.0
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 1.0, choke_vlv_op_3_prev: 1.0, choke_vlv_op_4_prev: 1.0
outputs: [1.0, 0.9994207472002731, 0.9994207472002731, 0.9994207472002731, 3457.9468617054763], clamped_outputs: [1.0, 0.9994207472002731, 0.9994207472002731, 0.9994207472002731, 3457.9468617054763]


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


Time step: 0
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.9994207472002731, choke_vlv_op_3: 0.9994207472002731, choke_vlv_op_4: 0.9994207472002731, pump_speed: 3457.9468617054763
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 1.0, choke_vlv_op_3_prev: 1.0, choke_vlv_op_4_prev: 1.0
outputs: [1.0, 0.998966065657424, 0.998952182173292, 0.998952182173292, 3457.912328993542], clamped_outputs: [1.0, 0.998966065657424, 0.998952182173292, 0.998952182173292, 3457.912328993542]
Time step: 30
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.998966065657424, choke_vlv_op_3: 0.998952182173292, choke_vlv_op_4: 0.998952182173292, pump_speed: 3457.912328993542
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 0.9994207472002731, choke_vlv_op_3_prev: 0.9994207472002731, choke_vlv_op_4_prev: 0.9994207472002731


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


outputs: [1.0, 0.998603526835904, 0.9985673216407579, 0.9985673216407579, 3457.8805426913605], clamped_outputs: [1.0, 0.998603526835904, 0.9985673216407579, 0.9985673216407579, 3457.8805426913605]


...

outputs: [1.0, 0.9867770097705018, 0.9727167162137381, 0.9110754184812873, 3449.574855660031], clamped_outputs: [1.0, 0.9867770097705018, 0.9727167162137381, 0.8854502656327451, 3449.574855660031]
Time step: 7140
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.9867770097705018, choke_vlv_op_3: 0.9727167162137381, choke_vlv_op_4: 0.8854502656327451, pump_speed: 3449.574855660031
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 0.9867731664865809, choke_vlv_op_3_prev: 0.9727124885587171, choke_vlv_op_4_prev: 0.8854469637481451
outputs: [1.0, 0.9867807249307229, 0.9727208157450592, 0.9110785926692662, 3449.5769276227475], clamped_outputs: [1.0, 0.9867807249307229, 0.9727208157450592, 0.885453439338595, 3449.5769276227475]


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


Time step: 7170
choke_vlv_op_1: 1.0, choke_vlv_op_2: 0.9867807249307229, choke_vlv_op_3: 0.9727208157450592, choke_vlv_op_4: 0.885453439338595, pump_speed: 3449.5769276227475
choke_vlv_op_1_prev: 1.0, choke_vlv_op_2_prev: 0.9867770097705018, choke_vlv_op_3_prev: 0.9727167162137381, choke_vlv_op_4_prev: 0.8854502656327451


┌ Warning: thread = 1 warning: parsed expected 438 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV /home/archanak/.julia/packages/CSV/LiiJM/src/file.jl:593


outputs: [1.0, 0.9867843119672439, 0.9727247871526802, 0.911081638251416, 3449.579025175781], clamped_outputs: [1.0, 0.9867843119672439, 0.9727247871526802, 0.8854564848656948, 3449.579025175781]
Time step: 7200


In [4]:
# EXTRACT ALL RESULTS FROM LEDAFLOW TO CSV FILE
include("extract_full_output.jl")

extract_full_output(lf_case_id, "caseC_trends.csv")

Process(`'/mnt/c/Program Files/Kongsberg/LedaFlow Engineering v2.11.271.018/softsh.exe' '/home/archanak/Kumaraswamy_2024_2027/Ongoing Work/2026_NPC_Workshop/ledaflow_extract.js'`, ProcessExited(0))